# Predictive Financial Time-Series Forecasting Using a Hybrid AI Framework

**End-to-End Predictive Analytics Project (KDD / Predictive Modelling workflow)**

| KDD / Modelling stage | Notebook section |
|---|---|
| 1. Problem identification | Section 1 |
| 2. Data selection & collection | Section 3 |
| 3. Data understanding (EDA) | Section 4 |
| 4. Pre-processing & transformation | Section 5 |
| 5. Data mining / modelling | Sections 6 – 7 |
| 6. Evaluation | Section 8 |
| 7. Interpretation & knowledge | Section 9 |

## 1. Problem Identification

**Business problem.** Investors, risk managers and portfolio analysts need an estimate of where a market index will close *tomorrow*. Financial series are difficult to forecast: they contain a (mostly) linear, autocorrelated component **and** non-linear, volatility-driven structure that a single classical model cannot capture.

**Predictive task.** Supervised, one-step-ahead forecasting of the next trading day's closing level (regression) and its direction (up / down) for two stock indices.

**Hybrid AI idea.** A classical statistical model (ARIMA) captures the linear structure of the series. A deep-learning model (LSTM) is then trained on the *residuals* of ARIMA, together with technical indicators, to learn the non-linear patterns that ARIMA misses:

$$\hat{y}_t = \hat{L}_t^{\,ARIMA} + \hat{N}_t^{\,LSTM}$$

**Objectives**
1. Build and compare five forecasting models on **two datasets** (NIFTY 50 and S&P 500).
2. Test whether the hybrid ARIMA-LSTM framework improves on each of its components and on a naïve random-walk benchmark.
3. Assess results with error metrics, directional accuracy and a statistical significance test (Diebold-Mariano).

**Models compared**

| Model | Role |
|---|---|
| Naïve (random walk) | Benchmark: tomorrow = today |
| ARIMA | Classical statistical model |
| Random Forest | Classical machine-learning model on technical indicators |
| LSTM | Deep-learning model on technical indicators |
| Hybrid ARIMA-LSTM (residual) | ARIMA + LSTM trained on ARIMA residuals only |
| Hybrid ARIMA-LSTM (residual + indicators) | ARIMA + LSTM trained on residuals and indicators (proposed framework) |

## 2. Setup

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy import stats

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

In [ ]:
# ---------------- Configuration ----------------
DATASETS = {
    "NIFTY 50": {"ticker": "^NSEI", "file": "data/NIFTY50.csv"},
    "S&P 500":  {"ticker": "^GSPC", "file": "data/SP500.csv"},
}
START, END = "2010-01-01", "2025-12-31"   # END is exclusive in yfinance

TRAIN_FRAC, VAL_FRAC = 0.70, 0.15          # remaining 15 % = test (chronological split)
WINDOW   = 20                              # look-back window (trading days) for LSTM
EPOCHS   = 60
BATCH    = 32
N_SEEDS  = 3                               # LSTM predictions are averaged over N_SEEDS runs (set 1 for speed)

## 3. Data Selection & Collection

Two independent datasets of daily OHLCV data from **Yahoo Finance** are used:

| Dataset | Ticker | Market |
|---|---|---|
| NIFTY 50 | `^NSEI` | India |
| S&P 500 | `^GSPC` | United States |

If the CSV files exist in `data/` they are loaded directly (the submitted repository includes them). Otherwise they are downloaded with `yfinance` and saved to `data/`.

In [ ]:
def load_dataset(name):
    cfg = DATASETS[name]
    if os.path.exists(cfg["file"]):
        df = pd.read_csv(cfg["file"], index_col=0, parse_dates=True)
    else:
        import yfinance as yf
        df = yf.download(cfg["ticker"], start=START, end=END, auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        os.makedirs(os.path.dirname(cfg["file"]), exist_ok=True)
        df.to_csv(cfg["file"])
    df = df[["Open", "High", "Low", "Close", "Volume"]].sort_index()
    df.index.name = "Date"
    return df

raw = {name: load_dataset(name) for name in DATASETS}
for name, df in raw.items():
    print(f"\n=== {name} ===")
    display(df.head(3))

## 4. Data Understanding (Exploratory Data Analysis)

In [ ]:
# 4.1 Basic inspection
rows = []
for name, df in raw.items():
    rows.append({
        "Dataset": name, "Ticker": DATASETS[name]["ticker"], "Rows": len(df),
        "Start": df.index.min().date(), "End": df.index.max().date(),
        "Missing values": int(df.isna().sum().sum()),
        "Duplicate dates": int(df.index.duplicated().sum()),
        "Zero-volume days": int((df["Volume"] == 0).sum()),
    })
display(pd.DataFrame(rows).set_index("Dataset"))
for name, df in raw.items():
    print(f"\n=== {name}: descriptive statistics ===")
    display(df.describe().T)

> **Note:** index volume is unreliable (often zero for NIFTY 50 in the early years), so **Volume is not used as a model feature**.

In [ ]:
# 4.2 Price level and rolling volatility
fig, ax = plt.subplots(2, 2, figsize=(14, 7))
for j, (name, df) in enumerate(raw.items()):
    ax[0, j].plot(df["Close"], lw=1)
    ax[0, j].set_title(f"{name} - closing level")
    r = np.log(df["Close"]).diff()
    ax[1, j].plot(r.rolling(21).std() * np.sqrt(252), color="darkred", lw=1)
    ax[1, j].set_title(f"{name} - 21-day rolling annualised volatility")
plt.tight_layout(); plt.show()

In [ ]:
# 4.3 Return distribution (log returns)
rows = []
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for j, (name, df) in enumerate(raw.items()):
    r = np.log(df["Close"]).diff().dropna()
    sns.histplot(r, bins=80, kde=True, ax=ax[j])
    ax[j].set_title(f"{name} - daily log-return distribution")
    jb = stats.jarque_bera(r)
    rows.append({"Dataset": name, "Mean": r.mean(), "Std": r.std(), "Skew": r.skew(),
                 "Excess kurtosis": r.kurt(), "Min": r.min(), "Max": r.max(), "Jarque-Bera p": jb.pvalue})
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index("Dataset").round(5))

In [ ]:
# 4.4 Stationarity (ADF test): H0 = unit root (non-stationary)
rows = []
for name, df in raw.items():
    lc = np.log(df["Close"]); r = lc.diff().dropna()
    rows.append({"Dataset": name,
                 "ADF p-value: price level": adfuller(df["Close"])[1],
                 "ADF p-value: log price": adfuller(lc)[1],
                 "ADF p-value: log return": adfuller(r)[1]})
display(pd.DataFrame(rows).set_index("Dataset").round(4))

Prices are non-stationary (cannot reject a unit root), while log-returns are stationary. This justifies using **one order of differencing (d = 1)** in ARIMA and **returns as the learning target** for the ML / DL models.

In [ ]:
# 4.5 Autocorrelation: returns (linear structure) vs squared returns (volatility clustering)
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
for j, (name, df) in enumerate(raw.items()):
    r = np.log(df["Close"]).diff().dropna()
    plot_acf(r, lags=30, ax=ax[0, j], title=f"{name} - ACF of returns")
    plot_acf(r ** 2, lags=30, ax=ax[1, j], title=f"{name} - ACF of squared returns")
plt.tight_layout(); plt.show()

In [ ]:
# 4.6 Relationship between the two datasets
rets = pd.concat({n: np.log(d["Close"]).diff() for n, d in raw.items()}, axis=1).dropna()
print("Overlapping trading days:", len(rets))
print("Correlation of daily returns (same day)   :", round(rets.iloc[:, 0].corr(rets.iloc[:, 1]), 3))
print("Correlation NIFTY(t) vs S&P(t-1)          :", round(rets.iloc[:, 0].corr(rets.iloc[:, 1].shift(1)), 3))

**EDA findings.** (1) Prices trend and are non-stationary; returns are stationary. (2) Returns show fat tails (high excess kurtosis) and are non-normal. (3) Returns have weak linear autocorrelation, but squared returns are strongly autocorrelated - *volatility clustering*, i.e. non-linear structure that a linear ARIMA model cannot represent. This is the motivation for the hybrid design.

## 5. Pre-processing & Transformation

Steps:
1. Remove duplicate dates, sort chronologically, forward-fill gaps in OHLC data.
2. Log-transform the closing price and compute log returns (the prediction target).
3. Engineer technical indicators (return lags, moving-average gaps, rolling volatility, RSI, MACD, high-low range).
4. **Shift all features by one day** so that the row for day *t* only contains information available at the close of day *t-1* (prevents look-ahead leakage).
5. **Chronological split:** 70 % train / 15 % validation / 15 % test. No shuffling.
6. Scalers are fitted on the training portion only.

In [ ]:
def rsi(close, n=14):
    d = close.diff()
    gain = d.clip(lower=0).rolling(n).mean()
    loss = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + gain / loss.replace(0, np.nan))

FEATURES = ["ret_1", "ret_2", "ret_3", "ret_5_mean", "ma_gap_5", "ma_gap_20",
            "vol_10", "vol_20", "rsi_14", "macd", "hl_range"]

def build_dataset(df):
    df = df[["Open", "High", "Low", "Close"]].copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.replace(0, np.nan).ffill().dropna()

    close = df["Close"]; logc = np.log(close); ret = logc.diff()

    ind = pd.DataFrame(index=df.index)                       # indicators known at close of day t
    ind["ret_1"] = ret
    ind["ret_2"] = ret.shift(1)
    ind["ret_3"] = ret.shift(2)
    ind["ret_5_mean"] = ret.rolling(5).mean()
    ind["ma_gap_5"] = close / close.rolling(5).mean() - 1
    ind["ma_gap_20"] = close / close.rolling(20).mean() - 1
    ind["vol_10"] = ret.rolling(10).std()
    ind["vol_20"] = ret.rolling(20).std()
    ind["rsi_14"] = rsi(close) / 100
    ind["macd"] = (close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()) / close
    ind["hl_range"] = (df["High"] - df["Low"]) / close
    X = ind.shift(1)                                         # <-- information up to t-1 only

    out = pd.concat([close.rename("close"), close.shift(1).rename("prev_close"),
                     ret.rename("ret"), X], axis=1).dropna()
    return out, logc, ret

D = {}
for name, df in raw.items():
    out, logc, ret_full = build_dataset(df)
    n = len(out)
    i_tr, i_va = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    D[name] = dict(out=out, logc=logc, ret_full=ret_full, n=n, i_tr=i_tr, i_va=i_va)

rows = []
for name, d in D.items():
    o = d["out"]
    for part, a, b in [("Train", 0, d["i_tr"]), ("Validation", d["i_tr"], d["i_va"]), ("Test", d["i_va"], d["n"])]:
        rows.append({"Dataset": name, "Split": part, "Rows": b - a,
                     "From": o.index[a].date(), "To": o.index[b - 1].date()})
display(pd.DataFrame(rows).set_index(["Dataset", "Split"]))
display(D["NIFTY 50"]["out"].head(3))

## 6. Modelling (I): Statistical & Classical ML Models

### 6.1 Naïve benchmark and ARIMA

* **Naïve:** tomorrow's close = today's close (predicted return = 0).
* **ARIMA(p, 1, q):** order selected by minimum AIC over p, q ∈ {0,…,3} using the **training data only**. The fitted parameters are then held fixed and applied to the whole series to obtain genuine **one-step-ahead** forecasts (each forecast uses only past observations).

In [ ]:
def select_arima(train_logc, max_p=3, max_q=3):
    best = None
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                res = ARIMA(train_logc, order=(p, 1, q)).fit()
            except Exception:
                continue
            if best is None or res.aic < best[0]:
                best = (res.aic, (p, 1, q), res)
    return best

PRED = {}      # predicted next-day log return, per dataset and model (all rows)
for name, d in D.items():
    out, logc, ret_full = d["out"], d["logc"], d["ret_full"]
    train_end_pos = logc.index.get_loc(out.index[d["i_tr"] - 1])
    aic, order, res = select_arima(logc.values[:train_end_pos + 1])
    d["arima_order"] = order

    applied = res.apply(logc.values)                         # fixed parameters, one-step-ahead filter
    fitted = pd.Series(applied.fittedvalues, index=logc.index)
    arima_ret_hat = fitted - logc.shift(1)                   # predicted log return
    arima_resid = ret_full - arima_ret_hat                   # residual (non-linear part)

    out["arima_ret_hat"] = arima_ret_hat.reindex(out.index)
    out["resid"] = arima_resid.reindex(out.index)
    out["resid_lag1"] = arima_resid.shift(1).reindex(out.index)

    PRED[name] = pd.DataFrame(index=out.index)
    PRED[name]["Naive"] = 0.0
    PRED[name]["ARIMA"] = out["arima_ret_hat"]
    print(f"{name}: selected ARIMA{order}  (AIC = {aic:.1f})")

In [ ]:
# Do ARIMA residuals still contain structure? (Ljung-Box on training residuals)
rows = []
for name, d in D.items():
    r = d["out"]["resid"].iloc[:d["i_tr"]]
    lb_lin = acorr_ljungbox(r, lags=[10])["lb_pvalue"].iloc[0]
    lb_sq  = acorr_ljungbox(r ** 2, lags=[10])["lb_pvalue"].iloc[0]
    rows.append({"Dataset": name, "ARIMA order": d["arima_order"],
                 "Ljung-Box p (residuals)": lb_lin, "Ljung-Box p (squared residuals)": lb_sq})
display(pd.DataFrame(rows).set_index("Dataset").round(4))

A small p-value for the *squared* residuals means that non-linear / volatility structure remains after ARIMA, which the LSTM component is intended to exploit.

### 6.2 Random Forest (classical ML)
Predicts the next-day log return from the 11 technical indicators. A small hyper-parameter grid is tuned on the **validation** set.

In [ ]:
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

RF_IMPORTANCE = {}
for name, d in D.items():
    o, i_tr, i_va = d["out"], d["i_tr"], d["i_va"]
    X, y = o[FEATURES].values, o["ret"].values
    best = None
    for depth in (3, 5, 8):
        for leaf in (10, 30):
            rf = RandomForestRegressor(n_estimators=300, max_depth=depth, min_samples_leaf=leaf,
                                       n_jobs=-1, random_state=SEED)
            rf.fit(X[:i_tr], y[:i_tr])
            score = rmse(y[i_tr:i_va], rf.predict(X[i_tr:i_va]))
            if best is None or score < best[0]:
                best = (score, depth, leaf, rf)
    _, depth, leaf, rf = best
    PRED[name]["Random Forest"] = rf.predict(X)
    RF_IMPORTANCE[name] = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
    print(f"{name}: best RF -> max_depth={depth}, min_samples_leaf={leaf}")

## 7. Modelling (II): Deep Learning and the Hybrid Framework

### 7.1 LSTM architecture
`LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16, ReLU) -> Dense(1)`

Input: the previous `WINDOW` = 20 trading days of features. Early stopping on the validation loss is used. The final prediction is the average over `N_SEEDS` independently initialised networks to reduce random-initialisation variance.

### 7.2 Three LSTM variants
| Variant | Input features | Target | Final forecast |
|---|---|---|---|
| **LSTM** | 11 indicators | next-day return | LSTM output |
| **Hybrid (residual)** | lagged ARIMA residual | ARIMA residual | ARIMA + LSTM |
| **Hybrid (residual + indicators)** | lagged ARIMA residual + 11 indicators | ARIMA residual | ARIMA + LSTM |

In [ ]:
def make_sequences(X, y, window):
    idx = np.arange(window - 1, len(X))
    Xs = np.stack([X[i - window + 1:i + 1] for i in idx])
    return Xs, y[idx], idx

def build_lstm(n_feat, window):
    m = keras.Sequential([
        keras.layers.Input(shape=(window, n_feat)),
        keras.layers.LSTM(64, return_sequences=True),
        keras.layers.Dropout(0.2),
        keras.layers.LSTM(32),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m

def fit_predict_lstm(X_raw, y_raw, i_tr, i_va, label):
    """Scale (train-only), build windows, train N_SEEDS networks, return de-scaled predictions for all rows."""
    sx = StandardScaler().fit(X_raw[:i_tr]);               Xs_all = sx.transform(X_raw)
    sy = StandardScaler().fit(y_raw[:i_tr, None]);         ys_all = sy.transform(y_raw[:, None]).ravel()
    Xw, yw, idx = make_sequences(Xs_all, ys_all, WINDOW)
    tr, va = idx < i_tr, (idx >= i_tr) & (idx < i_va)

    preds, hist = [], None
    for s in range(N_SEEDS):
        keras.utils.set_random_seed(SEED + s)
        model = build_lstm(Xw.shape[2], WINDOW)
        h = model.fit(Xw[tr], yw[tr], validation_data=(Xw[va], yw[va]), epochs=EPOCHS, batch_size=BATCH,
                      callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
                      verbose=0)
        preds.append(model.predict(Xw, verbose=0).ravel())
        hist = hist or h.history
        print(f"   {label} | seed {s}: {len(h.history['loss'])} epochs, best val loss = {min(h.history['val_loss']):.4f}")
    full = np.full(len(X_raw), np.nan)
    full[idx] = np.mean(preds, axis=0) * sy.scale_[0] + sy.mean_[0]
    return full, hist

HIST = {}
for name, d in D.items():
    o, i_tr, i_va = d["out"], d["i_tr"], d["i_va"]
    print(f"\n##### {name} #####")

    # (a) plain LSTM on indicators -> next-day return
    p, HIST[(name, "LSTM")] = fit_predict_lstm(o[FEATURES].values, o["ret"].values, i_tr, i_va, "LSTM")
    PRED[name]["LSTM"] = p

    # (b) hybrid, residual only -> ARIMA residual
    Xr = o[["resid_lag1"]].values
    p, HIST[(name, "Hybrid (residual)")] = fit_predict_lstm(Xr, o["resid"].values, i_tr, i_va, "Hybrid-res")
    PRED[name]["Hybrid ARIMA-LSTM (residual)"] = o["arima_ret_hat"].values + p

    # (c) hybrid, residual + indicators -> ARIMA residual
    Xh = np.column_stack([o["resid_lag1"].values, o[FEATURES].values])
    p, HIST[(name, "Hybrid (res+ind)")] = fit_predict_lstm(Xh, o["resid"].values, i_tr, i_va, "Hybrid-res+ind")
    PRED[name]["Hybrid ARIMA-LSTM (residual + indicators)"] = o["arima_ret_hat"].values + p

In [ ]:
# Training curves of the proposed hybrid model (first seed)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for j, name in enumerate(D):
    h = HIST[(name, "Hybrid (res+ind)")]
    ax[j].plot(h["loss"], label="train"); ax[j].plot(h["val_loss"], label="validation")
    ax[j].set_title(f"{name} - Hybrid LSTM loss"); ax[j].set_xlabel("epoch"); ax[j].set_ylabel("MSE (scaled)"); ax[j].legend()
plt.tight_layout(); plt.show()

## 8. Evaluation

Predicted returns are converted to a predicted closing level: $\hat{P}_t = P_{t-1}\,e^{\hat{r}_t}$. All models are evaluated on the **held-out test period only**.

| Metric | Meaning |
|---|---|
| RMSE / MAE | Error in index points |
| MAPE (%) | Mean absolute percentage error |
| Directional accuracy (%) | Share of days on which the predicted up/down direction is correct (not defined for the naïve model) |
| RMSE gain vs Naïve (%) | Improvement over the random-walk benchmark |
| Diebold-Mariano test | Tests whether the difference in squared forecast errors between two models is statistically significant |

In [ ]:
def evaluate(actual, pred, prev):
    err = actual - pred
    da = np.mean(np.sign(pred - prev) == np.sign(actual - prev)) * 100 if np.any(pred != prev) else np.nan
    return {"RMSE": np.sqrt(np.mean(err ** 2)), "MAE": np.mean(np.abs(err)),
            "MAPE (%)": np.mean(np.abs(err / actual)) * 100, "Directional accuracy (%)": da}

def dm_test(e1, e2):
    """One-step Diebold-Mariano test on squared errors. Negative statistic => model 1 more accurate."""
    dif = e1 ** 2 - e2 ** 2
    stat = dif.mean() / np.sqrt(dif.var(ddof=1) / len(dif))
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))

MODELS = ["Naive", "ARIMA", "Random Forest", "LSTM",
          "Hybrid ARIMA-LSTM (residual)", "Hybrid ARIMA-LSTM (residual + indicators)"]
RESULTS, TESTPRICE = {}, {}
for name, d in D.items():
    o, i_va = d["out"], d["i_va"]
    te = o.iloc[i_va:]
    actual, prev = te["close"].values, te["prev_close"].values
    P = pd.DataFrame({m: prev * np.exp(PRED[name][m].iloc[i_va:].values) for m in MODELS}, index=te.index)
    TESTPRICE[name] = (te["close"], P)
    tab = pd.DataFrame({m: evaluate(actual, P[m].values, prev) for m in MODELS}).T
    tab["RMSE gain vs Naive (%)"] = (1 - tab["RMSE"] / tab.loc["Naive", "RMSE"]) * 100
    RESULTS[name] = tab
    print(f"\n=== {name}  (test: {te.index[0].date()} to {te.index[-1].date()}, {len(te)} days) ===")
    display(tab.round(3))

In [ ]:
# Diebold-Mariano tests (squared price errors)
rows = []
for name in D:
    actual, P = TESTPRICE[name]
    HYB = "Hybrid ARIMA-LSTM (residual + indicators)"
    for other in ["Naive", "ARIMA", "LSTM", "Random Forest", "Hybrid ARIMA-LSTM (residual)"]:
        stat, p = dm_test((actual - P[HYB]).values, (actual - P[other]).values)
        rows.append({"Dataset": name, "Hybrid (res+ind) vs": other, "DM statistic": stat, "p-value": p,
                     "Significant at 5 %": "Yes" if p < 0.05 else "No",
                     "Hybrid better?": "Yes" if stat < 0 else "No"})
display(pd.DataFrame(rows).set_index(["Dataset", "Hybrid (res+ind) vs"]).round(4))

In [ ]:
# Model comparison charts
fig, ax = plt.subplots(2, 2, figsize=(15, 8))
short = {m: m.replace("Hybrid ARIMA-LSTM", "Hybrid") for m in MODELS}
for j, name in enumerate(D):
    t = RESULTS[name].rename(index=short)
    t["RMSE"].plot.bar(ax=ax[0, j], color=sns.color_palette("viridis", len(t)))
    ax[0, j].set_title(f"{name} - test RMSE (lower is better)"); ax[0, j].tick_params(axis="x", rotation=30)
    t["Directional accuracy (%)"].plot.bar(ax=ax[1, j], color=sns.color_palette("viridis", len(t)))
    ax[1, j].axhline(50, color="red", ls="--", lw=1, label="50 % (coin flip)"); ax[1, j].legend()
    ax[1, j].set_title(f"{name} - directional accuracy (%)"); ax[1, j].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Actual vs predicted (last 120 test days)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for j, name in enumerate(D):
    actual, P = TESTPRICE[name]
    a, p = actual.iloc[-120:], P.iloc[-120:]
    ax[j].plot(a, color="black", lw=2, label="Actual")
    ax[j].plot(p["ARIMA"], lw=1.2, label="ARIMA")
    ax[j].plot(p["LSTM"], lw=1.2, label="LSTM")
    ax[j].plot(p["Hybrid ARIMA-LSTM (residual + indicators)"], lw=1.5, label="Hybrid (res+ind)")
    ax[j].set_title(f"{name} - one-step-ahead forecasts"); ax[j].legend(); ax[j].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrices for direction (up / down) of the proposed hybrid and ARIMA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
fig, ax = plt.subplots(2, 2, figsize=(9, 7.5))
for j, name in enumerate(D):
    actual, P = TESTPRICE[name]
    prev = D[name]["out"]["prev_close"].iloc[D[name]["i_va"]:]
    y_true = (actual > prev).astype(int)
    for i, m in enumerate(["ARIMA", "Hybrid ARIMA-LSTM (residual + indicators)"]):
        y_pred = (P[m] > prev).astype(int)
        ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]), display_labels=["Down", "Up"]).plot(
            ax=ax[i, j], colorbar=False, cmap="Blues")
        ax[i, j].set_title(f"{name}\n{short[m]}", fontsize=9)
plt.tight_layout(); plt.show()

## 9. Interpretation of Results

In [ ]:
# 9.1 Which indicators does the Random Forest rely on?
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for j, name in enumerate(D):
    RF_IMPORTANCE[name].plot.barh(ax=ax[j], color="teal")
    ax[j].set_title(f"{name} - Random Forest feature importance")
plt.tight_layout(); plt.show()

In [ ]:
# 9.2 Residual diagnostics of the hybrid model (test period)
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
rows = []
HYB = "Hybrid ARIMA-LSTM (residual + indicators)"
for j, name in enumerate(D):
    actual, P = TESTPRICE[name]
    prev = D[name]["out"]["prev_close"].iloc[D[name]["i_va"]:]
    err_pct = (actual - P[HYB]) / prev * 100
    sns.histplot(err_pct, bins=50, kde=True, ax=ax[0, j]); ax[0, j].set_title(f"{name} - hybrid forecast error (% of price)")
    plot_acf(err_pct, lags=20, ax=ax[1, j], title=f"{name} - ACF of hybrid forecast errors")
    rows.append({"Dataset": name, "Mean error (%)": err_pct.mean(), "Std error (%)": err_pct.std(),
                 "Ljung-Box p (errors)": acorr_ljungbox(err_pct, lags=[10])["lb_pvalue"].iloc[0]})
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index("Dataset").round(4))

In [ ]:
# 9.3 Automatic summary of the experiment (generated from the numbers above)
for name in D:
    t = RESULTS[name]
    best_rmse = t["RMSE"].idxmin()
    best_da = t["Directional accuracy (%)"].idxmax()
    hyb = t.loc["Hybrid ARIMA-LSTM (residual + indicators)"]
    print(f"{name}:")
    print(f"  Lowest test RMSE          : {best_rmse}  ({t.loc[best_rmse, 'RMSE']:.2f})")
    print(f"  Highest directional acc.  : {best_da}  ({t.loc[best_da, 'Directional accuracy (%)']:.2f} %)")
    print(f"  Hybrid (res+ind) vs Naive : RMSE gain {hyb['RMSE gain vs Naive (%)']:.2f} %, "
          f"direction {hyb['Directional accuracy (%)']:.2f} %")
    print(f"  Hybrid vs ARIMA (RMSE)    : {(1 - hyb['RMSE'] / t.loc['ARIMA', 'RMSE']) * 100:.2f} % change\n")

In [ ]:
# 9.4 Example: forecasts for the last 5 test days (NIFTY 50)
actual, P = TESTPRICE["NIFTY 50"]
example = pd.DataFrame({"Actual close": actual,
                        "ARIMA": P["ARIMA"],
                        "Hybrid (res+ind)": P["Hybrid ARIMA-LSTM (residual + indicators)"]}).tail(5)
display(example.round(2))

### 9.5 Discussion and conclusions

*(Read these points against the tables and charts produced by your own run - do not copy numbers that you did not obtain.)*

1. **Benchmark difficulty.** Daily index levels are close to a random walk, so the naïve model is a very strong baseline in absolute-error terms. If the AIC selects ARIMA(0,1,0) for a dataset, ARIMA *is* a random walk and gives identical forecasts to the naïve model (its directional accuracy is then undefined) - in that case the hybrid's LSTM part is the only source of any improvement. Check the *RMSE gain vs Naïve* column and the Diebold-Mariano p-values to see whether any model beats it by a statistically meaningful margin, rather than judging by RMSE alone.
2. **Value of the hybrid.** Compare *Hybrid* with *ARIMA* and with *LSTM* separately: the hybrid is only justified if it improves on both of its components. The ablation between *Hybrid (residual)* and *Hybrid (residual + indicators)* shows whether the extra indicators add information.
3. **Direction vs. magnitude.** A model can have a similar RMSE to the benchmark and still provide useful (or useless) directional signals. Directional accuracy close to 50 % indicates little directional skill.
4. **Two markets.** Differences between NIFTY 50 and S&P 500 results indicate how stable the framework is across markets with different volatility regimes.
5. **Interpretability.** Random Forest importances indicate which indicators (volatility, momentum, recent returns) carry predictive information.

### Limitations and future work
* Daily index data only; no macroeconomic, sentiment or news variables.
* Transaction costs, slippage and a trading-strategy back-test are not included - forecast accuracy does not equal profitability.
* ARIMA parameters are fixed after training (no periodic re-estimation); walk-forward re-training is a natural extension.
* Possible extensions: GARCH volatility features, Transformer / attention models, ARIMA-GARCH-LSTM, walk-forward cross-validation, additional markets or assets (e.g. Bitcoin, gold).